### _**01. Raw to Landing**_

This section handles the initial ingestion of daily transaction files from the **Azure Data Lake Storage (ADLS) Gen2** raw zone into the Spark environment.

### <mark><u>_**Configuration**_</u></mark>
* **Source Path**: `abfss://fabric-project@fabricproject1234...`
* **File Pattern**: `LMS_MM-DD-YYYY.csv`
* **Ingestion Logic**:
    1. Define the storage account and container variables.
    2. Construct the full path using the `today_file` parameter.
    3. Read the CSV using `spark.read` with `header=True` and `inferSchema=True`.

> **Note:** The `processed_Date` variable is used for metadata tracking to ensure we know exactly when this specific file was picked up by the pipeline.

In [1]:
today_file = 'file ' #'LMS_09-02-2023.csv'
processed_Date = '9999-99-99'#'2026-03-20'

StatementMeta(, 1ecfff97-abc7-4de2-a643-f65e08fdcf79, 3, Finished, Available, Finished, False)

In [3]:
account_name = 'fabricproject12345' # fill in your primary account name 
container_name = 'container111' # fill in your container name 
relative_path = 'raw' # fill in your relative folder path 

adls_path = 'abfss://%s@%s.dfs.core.windows.net/%s' % (container_name, account_name, relative_path) 

print('Source storage account path is ', adls_path)

StatementMeta(, 1ecfff97-abc7-4de2-a643-f65e08fdcf79, 5, Finished, Available, Finished, False)

Source storage account path is  abfss://container111@fabricproject12345.dfs.core.windows.net/raw


### _**Reading csv file of Today**_

In [9]:
latest_path = f"{adls_path}/{today_file}"
print(latest_path)

StatementMeta(, 5ab99586-de19-408b-ac77-ac016ea02761, 11, Finished, Available, Finished, False)

abfss://container111@fabricproject12345.dfs.core.windows.net/raw/LMS_09-01-2023.csv


In [11]:
latest_path = f"{adls_path}/{today_file}"
df = spark.read.csv(path = latest_path, header=True, inferSchema=True)

display(df.limit(5))

StatementMeta(, 5ab99586-de19-408b-ac77-ac016ea02761, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7ea0bddb-94ac-4b2a-986a-4da76a8df9fa)

In [4]:
from pyspark.sql.functions import lit
latest_path = f"{adls_path}/{today_file}"
df = spark.read.csv(path= latest_path,header=True,inferSchema=True)

if df.count() > 1:
    print("The file has data.") #prevent with file with no data
    
    df_new = df.withColumn("Processing_Date",lit(processed_Date))
    df_new.write.format('csv').option('header','true').partitionBy('Processing_Date').mode('append').save('abfss://container111@fabricproject12345.dfs.core.windows.net/landing/')

    print("Data written to landing zone successfully !")

else:
    print('This file contains only header row and no data.')

StatementMeta(, 1ecfff97-abc7-4de2-a643-f65e08fdcf79, 6, Finished, Available, Finished, False)

The file has data.
Data written to landing zone successfully !
